### CS 378: Automated Planning for Robots
# HW 3: Modeling Temporal Planning Problems in Unified Planning Framework
### Spring 2026
### Lecturer: Erez Karpas
### TA: Talal Ayman

In this homework assignment you will adapt the planning problem from HW1 to include action durations and allow robot to operate concurrently, by modeling the problem as a temporal planning problem.

As in HW1, you will use the [Unified Planning Framework](https://github.com/aiplan4eu/unified-planning), so the first step is to make sure the unified planning framework is installed.

We must also make sure to import the package.

In [1]:
%pip install unified-planning[tamer]

import unified_planning
from unified_planning.shortcuts import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 1.5 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [up-tamer]
Note: you may need to restart the kernel to use updated packages.


We will now create a demo temporal planning problem - the match cellar problem we saw in class.

In [2]:
Match = UserType('Match')
Fuse = UserType('Fuse')

handfree = Fluent('handfree')
light = Fluent('light')
match_used = Fluent('match_used', BoolType(), m=Match)
fuse_mended = Fluent('fuse_mended', BoolType(), f=Fuse)

problem = Problem('MatchCellar')

problem.add_fluent(handfree)
problem.add_fluent(light)
problem.add_fluent(match_used, default_initial_value=False)
problem.add_fluent(fuse_mended, default_initial_value=False)

problem.set_initial_value(light, False)
problem.set_initial_value(handfree, True)

fuses = [Object(f'f{i}', Fuse) for i in range(3)]
matches = [Object(f'm{i}', Match) for i in range(3)]

problem.add_objects(fuses)
problem.add_objects(matches)

light_match = DurativeAction('light_match', m=Match)
m = light_match.parameter('m')
light_match.set_fixed_duration(6)
light_match.add_condition(StartTiming(), Not(match_used(m)))
light_match.add_effect(StartTiming(), match_used(m), True)
light_match.add_effect(StartTiming(), light, True)
light_match.add_effect(EndTiming(), light, False)
problem.add_action(light_match)

mend_fuse = DurativeAction('mend_fuse', f=Fuse)
f = mend_fuse.parameter('f')
mend_fuse.set_fixed_duration(5)
mend_fuse.add_condition(StartTiming(), handfree)
mend_fuse.add_condition(ClosedTimeInterval(StartTiming(), EndTiming()), light)
mend_fuse.add_effect(StartTiming(), handfree, False)
mend_fuse.add_effect(EndTiming(), fuse_mended(f), True)
mend_fuse.add_effect(EndTiming(), handfree, True)
problem.add_action(mend_fuse)

for f in fuses:
  problem.add_goal(fuse_mended(f))

print(problem)

problem name = MatchCellar

types = [Match, Fuse]

fluents = [
  bool handfree
  bool light
  bool match_used[m=Match]
  bool fuse_mended[f=Fuse]
]

actions = [
  durative action light_match(Match m) {
    duration = [6, 6]
    conditions = [
      [start]:
        (not match_used(m))
    ]
    effects = [
      start:
        match_used(m) := true:
        light := true:
      end:
        light := false:
    ]
    simulated effects = [
    ]
  }
  durative action mend_fuse(Fuse f) {
    duration = [5, 5]
    conditions = [
      [start]:
        handfree
      [start, end]:
        light
    ]
    effects = [
      start:
        handfree := false:
      end:
        fuse_mended(f) := true:
        handfree := true:
    ]
    simulated effects = [
    ]
  }
]

objects = [
  Match: [m0, m1, m2]
  Fuse: [f0, f1, f2]
]

initial fluents default = [
  bool match_used[m=Match] := false
  bool fuse_mended[f=Fuse] := false
]

initial values = [
  light := false
  handfree := true
]

goals = 

We can now call solve our problem, by using the OneShotPlanner mode.

In [3]:
with OneshotPlanner(name='tamer') as planner:
    result = planner.solve(problem)
    plan = result.plan
    if plan is not None:
        print(f"{planner.name} returned:")
        for start, action, duration in plan.timed_actions:
            print(f"{float(start)}: {action} [{float(duration)}]")
    else:
        print("No plan found.")

NOTE: To disable printing of planning engine credits, add this line to your code: `up.shortcuts.get_environment().credits_stream = None`
  *** Credits ***
  * In operation mode `OneshotPlanner` at line 1 of `/tmp/ipykernel_8199/1287088451.py`, you are using the following planning engine:
  * Engine name: Tamer
  * Developers:  FBK Tamer Development Team
  * Description: Tamer offers the capability to generate a plan for classical, numerical and temporal problems.
  *              For those kind of problems tamer also offers the possibility of validating a submitted plan.

Tamer returned:
0.0: light_match(m1) [6.0]
0.01: mend_fuse(f1) [5.0]
6.01: light_match(m2) [6.0]
6.02: mend_fuse(f2) [5.0]
12.02: light_match(m0) [6.0]
12.03: mend_fuse(f0) [5.0]


Now that we have made sure everything is working, we can start describing the homework assignment

##  Homework Assignment: Modeling Control of Home Service Robots as a Temporal Planning Problem

### Part1: Modeling
In this homework assignment, you will model a planning problem which can plan for a set of home service robots. This is the same as HW1, except for the addition of action durations.

We have different kinds of robots:
* Mobile manipulator - a mobile robot which can move between rooms, and tidy up a room. However, the mobile manipulator has dirt on its wheelrs, so after a mobile manipulator tidies up a room, it is no longer clean. Moving between rooms takes 30 seconds, tidying up a room takes 300 seconds.
* Mobile vacuum - a mobile robot which can move between rooms, and vacuum the room it is in. A room can only be vacuumed after it has been tidied up. After a room has been vacuumed it is clean. Moving between rooms takes 30 seconds, vacuuming a room takes 400 seconds.

Finally, rooms are connected according to some connectivity map.
The goal is to have all rooms clean and tidy.

Your assignment is to model the given problem using the Unified Planning Framework.
You will need to solve the problem for the following maps:

Map 1:
* 2 connected rooms (rm1 and rm2)
* 1 mobile manipulator, starts in rm1
* 1 vacuum, starts in rm1
* both rooms start not clean and not tidy

Map2:
* 4 rooms - nw, ne, sw, se (4 corners). Each room is connected to two adjacent rooms, and not to the one in the opposite corner (for example, nw is connected to ne and sw, but not to se).
* 1 mobile manipulator, starts in sw
* 1 vacuum, starts in ne
* all rooms start not clean and not tidy

Map3:
* 9 rooms: corridor (which is a room), which is connected to 8 different rooms (rm1 -- rm8) 
* 1 mobile manipulator, starts in corridor
* 1 vacuum, starts in corridor
* rooms rm1 -- rm4 start tidy but not clean, rooms rm5 -- rm8 start clean but not tidy.

In the following code block, you will need to define 3 functions: ``prob1(), prob2(), prob3()`` 
Each of these functions should return a problem object which corresponds to the map described above.

The code block below will call your functions to generate the problem, and then call a planner to solve them and print the solutions.


In [ ]:
def prob1():
    return None

def prob2():
    return None

def prob3():
    return None

The code below will call your functions to generate the problems, and then try to solve them using Pyperplan.

In [ ]:
from unified_planning.engines import PlanGenerationResultStatus

def solve(prob):
    planner = OneshotPlanner(name='tamer')
    result = planner.solve(prob)
    if result.status in [PlanGenerationResultStatus.SOLVED_SATISFICING,
                         PlanGenerationResultStatus.SOLVED_OPTIMALLY]:
        print("SOLVED")
        for start, action, duration in plan.timed_actions:
            print(f"{float(start)}: {action} [{float(duration)}]")
    else:
        print("NOT SOLVED")
    

print("Map1")
solve(prob1())

print("Map2")
solve(prob2())

print("Map3")
solve(prob3())


### Results
Map1
SOLVED
0.0: tidy_room(mm1_map1, r1) [300.0]
0.0: move_room_manip(mm1_map1, r1, r2) [30.0]
30.01: tidy_room(mm1_map1, r2) [300.0]
300.01: move_room_vac(mv1_map1, r1, r2) [30.0]
300.01: clean_room(mv1_map1, r1) [400.0]
330.02: clean_room(mv1_map1, r2) [400.0]

Map2
SOLVED
0.0: move_room_manip(mm1_map2, sw, nw) [30.0]
0.0: tidy_room(mm1_map2, sw) [300.0]
30.01: tidy_room(mm1_map2, nw) [300.0]
30.01: move_room_manip(mm1_map2, nw, ne) [30.0]
60.02: move_room_manip(mm1_map2, ne, se) [30.0]
60.02: tidy_room(mm1_map2, ne) [300.0]
90.03: tidy_room(mm1_map2, se) [300.0]
360.03: clean_room(mv1_map2, ne) [400.0]
360.03: move_room_vac(mv1_map2, ne, nw) [30.0]
390.04: move_room_vac(mv1_map2, nw, sw) [30.0]
390.04: clean_room(mv1_map2, nw) [400.0]
420.05: clean_room(mv1_map2, sw) [400.0]
420.05: move_room_vac(mv1_map2, sw, se) [30.0]
450.06: clean_room(mv1_map2, se) [400.0]

Map3
SOLVED
0.0: move_room_manip(mm1_map3, corridor, rm2) [30.0]
0.0: move_room_vac(mv1_map3, corridor, rm3) [30.0]
30.1: tidy_room(mm1_map3, rm2) [300.0]
30.2: move_room_manip(mm1_map3, rm2, corridor) [30.0]
60.3: move_room_manip(mm1_map3, corridor, rm3) [30.0]
90.4: tidy_room(mm1_map3, rm3) [300.0]
90.6: tidy_room(mm1_map3, rm3) [300.0]
90.7: move_room_manip(mm1_map3, rm3, corridor) [30.0]
120.8: move_room_manip(mm1_map3, corridor, rm8) [30.0]
150.9: tidy_room(mm1_map3, rm8) [300.0]
151.0: move_room_manip(mm1_map3, rm8, corridor) [30.0]
181.1: move_room_manip(mm1_map3, corridor, rm7) [30.0]
211.2: tidy_room(mm1_map3, rm7) [300.0]
211.3: move_room_manip(mm1_map3, rm7, corridor) [30.0]
241.4: move_room_manip(mm1_map3, corridor, rm6) [30.0]
271.5: tidy_room(mm1_map3, rm6) [300.0]
271.6: move_room_manip(mm1_map3, rm6, corridor) [30.0]
301.7: move_room_manip(mm1_map3, corridor, rm1) [30.0]
331.8: move_room_manip(mm1_map3, rm1, corridor) [30.0]
361.9: move_room_manip(mm1_map3, corridor, rm5) [30.0]
390.5: clean_room(mv1_map3, rm3) [400.0]
390.6: move_room_vac(mv1_map3, rm3, corridor) [30.0]
392.0: tidy_room(mm1_map3, rm5) [300.0]
392.2: tidy_room(mm1_map3, rm5) [300.0]
420.7: move_room_vac(mv1_map3, corridor, rm8) [30.0]
451.0: clean_room(mv1_map3, rm8) [400.0]
451.1: move_room_vac(mv1_map3, rm8, corridor) [30.0]
481.2: move_room_vac(mv1_map3, corridor, rm1) [30.0]
511.3: clean_room(mv1_map3, rm1) [400.0]
511.4: move_room_vac(mv1_map3, rm1, corridor) [30.0]
541.5: move_room_vac(mv1_map3, corridor, rm4) [30.0]
571.6: clean_room(mv1_map3, rm4) [400.0]
571.7: move_room_vac(mv1_map3, rm4, corridor) [30.0]
601.8: move_room_vac(mv1_map3, corridor, rm5) [30.0]
692.1: clean_room(mv1_map3, rm5) [400.0]
692.2: move_room_vac(mv1_map3, rm5, corridor) [30.0]
722.3: move_room_vac(mv1_map3, corridor, rm2) [30.0]
752.4: clean_room(mv1_map3, rm2) [400.0]
752.5: move_room_vac(mv1_map3, rm2, corridor) [30.0]
782.6: move_room_vac(mv1_map3, corridor, rm7) [30.0]
812.7: clean_room(mv1_map3, rm7) [400.0]
812.8: move_room_vac(mv1_map3, rm7, corridor) [30.0]
842.9: move_room_vac(mv1_map3, corridor, rm6) [30.0]
873.0: clean_room(mv1_map3, rm6) [400.0]

### Part 2: Questions

Please answer each question in the markdown cell below it.
If your answer can be justified using some python code, feel free to also add a code block.

#### Q1

Are the solutions returned by the planner for all problems makespan optimal (meaning, do they take the shortest time possible)?
If so, explain.
If not, give a shorter plan to one of the problems.


#### Answer 1

Fill in your answer here...




#### Q2

Above we simplified the problem by assuming it takes 30 seconds to move between rooms. How would you solve the problem if the time to move between rooms was relative to the distance the robot needs to travel? Assume the robot moves at a constant, known, velocity. How would you compute the time to travel between a given pair of rooms? Think about temporal uncertainty, and how it can be avoided.

#### Answer 2

Fill in your answer here...